# Calibration — Baseline ISO 52016-1 on Apt 305

Calibrates the unmodified ISO 52016-1 engine on Apt 305, 50 Barry St, Carlton.

This notebook validates that the baseline engine produces consistent, reproducible results for the reference case using a fixed weather file.

**Building:** Single 20 m² Melbourne apartment, one exposed (west) facade, five conditioned neighbours  
**Weather:** Charlton, Victoria (2.0° from central Melbourne)  
**Engine:** Unmodified ISO 52016-1 (baseline)  
**Validation:** Reproducibility check — same engine, building, and weather must always give identical results

## 1. Clone and setup

In [ ]:
import os, shutil

REPO = "https://github.com/samiraghafarigousheh-sys/AIB.git"
BRANCH = "claude/pybuildingenergy-baseline-anjro8"

# Private repo? Uncomment and paste a token with `repo` scope:
# TOKEN = "ghp_xxx"
# REPO = f"https://{TOKEN}@github.com/samiraghafarigousheh-sys/AIB.git"

if os.path.isdir("AIB"):
    shutil.rmtree("AIB")

!git clone --quiet --branch $BRANCH $REPO AIB
%cd AIB
!git fetch --quiet origin '+refs/heads/*:refs/remotes/origin/*'
print("✓ Repository cloned")

In [ ]:
# Install dependencies
!pip install -q -r pybuildingenergy/requirements.txt
!git config user.email "colab@example.com"
!git config user.name "Colab"
print("✓ Dependencies installed")

## 2. Pinned weather file

In [ ]:
import glob, sys
sys.path.insert(0, "examples")

EPW = sorted(glob.glob("weather_cache/*.epw"))[0]

from weather_melbourne import read_epw_site, site_offset_deg
lat, lon, city = read_epw_site(EPW)
offset = site_offset_deg(lat, lon)

print(f"Weather file  : {EPW}")
print(f"Station       : {city}  (lat {lat}, lon {lon})")
print(f"Offset from central Melbourne: {offset:.2f}°")
print()
print("✓ Weather pinned to single EPW — all runs use identical weather")

## 3. Run the baseline engine (first calibration run)

In [ ]:
from apt305_building import build_bui
from pybuildingenergy.source.utils import ISO52016
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI

# Build and validate
bui = build_bui()
building, _ = sanitize_and_validate_BUI(bui, fix=True)

print(f"Building: {building['building']['name']}")
print(f"Floor area: {building['building']['net_floor_area']} m²")
print(f"Adjacent zones: {building['building']['number_adj_zone']}")
print()

# Run engine
result = ISO52016.Temperature_and_Energy_needs_calculation(
    building,
    weather_source="epw",
    path_weather_file=EPW,
)
annual = result[1] if isinstance(result, tuple) else None

if annual is None:
    raise RuntimeError("Failed to get annual results")

import pandas as pd
print("✓ Baseline engine run complete")

## 4. Calibration results — baseline metrics

In [ ]:
import pandas as pd

# Extract key annual metrics
metrics = {
    "Heating need (kWh)": float(pd.to_numeric(annual["Q_H_annual_kWh"], errors="coerce").iloc[0]),
    "Cooling need (kWh)": float(pd.to_numeric(annual["Q_C_annual_kWh"], errors="coerce").iloc[0]),
    "Solar gains (kWh)": float(pd.to_numeric(annual["Q_solar_gains_kWh"], errors="coerce").iloc[0]),
    "Window transm. loss (kWh)": float(pd.to_numeric(annual["Q_tr_window_loss_kWh"], errors="coerce").iloc[0]),
    "Opaque transm. loss (kWh)": float(pd.to_numeric(annual["Q_tr_opaque_loss_kWh"], errors="coerce").iloc[0]),
    "Total transm. loss (kWh)": float(pd.to_numeric(annual["Q_tr_total_loss_kWh"], errors="coerce").iloc[0]),
}

calib_df = pd.DataFrame([
    (label, f"{value:,.2f}")
    for label, value in metrics.items()
], columns=["Metric", "Value"])

print("="*60)
print("BASELINE CALIBRATION RESULTS — Apt 305")
print("="*60)
display(calib_df)
print()
print(f"Total annual energy (heating + cooling): {metrics['Heating need (kWh)'] + metrics['Cooling need (kWh)']:,.2f} kWh")

## 5. Reproducibility check — run again and compare

In [ ]:
# Run the engine a second time with identical inputs
result2 = ISO52016.Temperature_and_Energy_needs_calculation(
    building,
    weather_source="epw",
    path_weather_file=EPW,
)
annual2 = result2[1] if isinstance(result2, tuple) else None

# Extract metrics from second run
metrics2 = {
    "Heating need (kWh)": float(pd.to_numeric(annual2["Q_H_annual_kWh"], errors="coerce").iloc[0]),
    "Cooling need (kWh)": float(pd.to_numeric(annual2["Q_C_annual_kWh"], errors="coerce").iloc[0]),
    "Solar gains (kWh)": float(pd.to_numeric(annual2["Q_solar_gains_kWh"], errors="coerce").iloc[0]),
    "Window transm. loss (kWh)": float(pd.to_numeric(annual2["Q_tr_window_loss_kWh"], errors="coerce").iloc[0]),
    "Opaque transm. loss (kWh)": float(pd.to_numeric(annual2["Q_tr_opaque_loss_kWh"], errors="coerce").iloc[0]),
    "Total transm. loss (kWh)": float(pd.to_numeric(annual2["Q_tr_total_loss_kWh"], errors="coerce").iloc[0]),
}

# Compare
TOL = 1e-6  # Tolerance for identical runs
comparison = []
all_match = True

for label in metrics.keys():
    val1 = metrics[label]
    val2 = metrics2[label]
    denom = max(abs(val1), abs(val2), 1e-12)
    rel_diff = abs(val1 - val2) / denom
    match = rel_diff <= TOL
    all_match &= match
    
    comparison.append({
        "Metric": label,
        "Run 1": f"{val1:,.6f}",
        "Run 2": f"{val2:,.6f}",
        "Rel. diff": f"{rel_diff:.2e}",
        "Status": "✓" if match else "✗"
    })

comp_df = pd.DataFrame(comparison)
print("\n" + "="*80)
print("REPRODUCIBILITY TEST — Same engine, building, weather")
print("="*80)
display(comp_df)
print()
if all_match:
    print("✓ PASS  Both runs give identical results (rel. diff < 1e-6)")
    print("        Baseline calibration is reproducible and reliable.")
else:
    print("✗ FAIL  Results diverged between runs")
    print("        Check for non-determinism or numerical instability.")

## 6. Calibration validation — store baseline reference

In [ ]:
import json
from pathlib import Path

# Store calibration reference
calib_ref = {
    "engine": "ISO52016 baseline (unmodified)",
    "building": "Apt 305, 50 Barry St, Carlton",
    "weather": "AUS_VIC_Charlton.948390_TMYx.2009-2023.epw",
    "weather_offset_deg": offset,
    "metrics": metrics,
    "timestamp": pd.Timestamp.now().isoformat(),
    "reproducible": all_match,
}

output_dir = Path("results/calibration_baseline")
output_dir.mkdir(parents=True, exist_ok=True)

ref_file = output_dir / "baseline_calibration_reference.json"
ref_file.write_text(json.dumps(calib_ref, indent=2), encoding="utf-8")

print(f"✓ Calibration reference saved: {ref_file}")
print()
print("This reference establishes the GOLDEN STANDARD for this building.")
print("Future runs should always match these values (to numerical precision).")

## 7. Summary

In [ ]:
print("\n" + "="*70)
print("CALIBRATION SUMMARY")
print("="*70)
print(f"\nBuilding: Apt 305 — 20 m², Melbourne, one exposed facade")
print(f"Engine:   ISO 52016-1 (unmodified baseline)")
print(f"Weather:  Charlton TMYx ({offset:.1f}° from central Melbourne)")
print(f"\nBaseline Results:")
print(f"  Heating:  {metrics['Heating need (kWh)']:>10,.2f} kWh/yr")
print(f"  Cooling:  {metrics['Cooling need (kWh)']:>10,.2f} kWh/yr")
print(f"  Total:    {metrics['Heating need (kWh)'] + metrics['Cooling need (kWh)']:>10,.2f} kWh/yr")
print(f"\nReproducibility: {'✓ PASS' if all_match else '✗ FAIL'}")
print(f"\nThis calibration establishes a fixed reference point for:")
print(f"  • Validating future engine modifications")
print(f"  • Ensuring consistent results on repeated runs")
print(f"  • Comparing against EnergyPlus and other benchmarks")
print("\n" + "="*70)